# Neural TSP - n=15 (RL + size comparison)

Trains the Pointer Network with **reinforcement learning at n=15**, then compares the model
— decoded with **Active Search** — against the **exact optimum (Held-Karp)** and the
**heuristics (Nearest Neighbor, 2-Opt)** across **n = 10, 12, 15**, plotted as mean +/- std.

**Supervised pretraining is intentionally skipped.** Held-Karp computes exact optimal tours,
but at n=15 it costs ~0.4s/instance, so 100k labels would take hours. RL needs no labels -
the model learns directly from the tour-length reward.

**Why the size comparison uses only 50 instances:** Active Search is *per-instance iterative*
(~5000 gradient steps per instance), so 1000 instances would take ~tens of hours. Held-Karp /
NN / 2-Opt are evaluated on the *same* 50 instances for a fair mean/std.

Estimated total on a T4: ~10-20 min (train) + ~5 min (standard eval) + ~1.5-2 h (size comparison).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1 - Copy project & install deps

Also ensures `data/` exists: empty directories get dropped during Drive upload, which is the
bug that broke data generation before.

In [ ]:
# Copy project to Colab local storage (faster than reading from Drive)
!cp -r /content/drive/MyDrive/neural-tsp /content/neural-tsp
%cd /content/neural-tsp

!pip install torch numpy matplotlib -q

# FIX: empty data/ is dropped on upload - recreate it so the C++ cells can write into it
import os
os.makedirs('/content/neural-tsp/data', exist_ok=True)
print('Project copied, data/ ready.')

## 2 - Generate n=15 data (+ comparison sets at n=10/12/15)

Only the RL training set and eval set are needed for training (no supervised labels).
We also generate the three **comparison** raw sets at n=10/12/15 (100 instances each -
`compare_sizes.py` uses the first 50) for the Active-Search-vs-optimal study in step 8.

In [ ]:
%%bash
cd /content/neural-tsp/cpp
set -e

g++ -std=c++17 -O2 generate_data.cpp -o generate_data

mkdir -p ../data   # belt-and-suspenders: ensure ../data exists before writing

# RL training set + eval set at n=15
./generate_data 100000 15 ../data/train_rl.txt
./generate_data 1000 15 ../data/eval.txt

# Comparison raw sets at n=10/12/15 (compare_sizes.py uses the first 50 of each)
for n in 10 12 15; do
  ./generate_data 100 $n ../data/comp_n${n}.txt
done

ls -la ../data

## 3 - Save generated data to Drive

Backup so you don't regenerate next time.

In [ ]:
import shutil

shutil.copytree('/content/neural-tsp/data', '/content/drive/MyDrive/neural-tsp/data', dirs_exist_ok=True)
print('Data saved to Google Drive.')

## 4 - RL Actor-Critic training (n=15, from scratch)

Policy-gradient training with a learned critic baseline. `train_rl.py` reads `n` from the data
header, so n=15 needs no code change. Checkpoints are saved every 5 epochs and it auto-resumes
from the latest.

Any stale `model.pt` (e.g. an n=12 supervised stub) is removed first so the actor trains
genuinely from scratch.

~5-10 min on T4.

In [ ]:
%cd /content/neural-tsp/python

# Train from scratch. Drop any stale supervised model.pt so the actor isn't
# warm-started on the wrong graph size.
import os
if os.path.exists('model.pt'):
    os.remove('model.pt')
    print('Removed stale model.pt - training actor from scratch.')

%run train_rl.py

## 5 - Save RL models & checkpoints to Drive

Backup in case Colab disconnects.

In [ ]:
import shutil, os

for f in ['actor.pt', 'critic.pt']:
    src = f'/content/neural-tsp/python/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/drive/MyDrive/neural-tsp/python/{f}')

ckpt_dir = '/content/neural-tsp/python/checkpoints'
if os.path.exists(ckpt_dir):
    shutil.copytree(ckpt_dir, '/content/drive/MyDrive/neural-tsp/python/checkpoints', dirs_exist_ok=True)

print('Models and checkpoints saved to Google Drive.')

## 6 - Evaluate neural methods (greedy / sampling / active search)

Standard evaluation on the 1000-instance n=15 eval set. Greedy + Active Search (first 5) are
fast; the Sampling grid (N=1280 over 1000 instances) is the slow part - interrupt after Greedy
+ Active Search if you just want a quick check.

In [ ]:
%cd /content/neural-tsp/python
%run eval_rl.py

## 7 - Evaluate classical baselines

Nearest-Neighbor and 2-Opt on the n=15 eval set, for comparison.

In [ ]:
%%bash
cd /content/neural-tsp/cpp
set -e
g++ -std=c++17 -O2 baselines.cpp -o baselines
./baselines < ../data/eval.txt

## 8 - Size comparison: Active Search vs Held-Karp & heuristics (n=10/12/15)

Compares the **trained n=15 model (decoded with Active Search)** against the **exact optimum
(Held-Karp)** and the **heuristics (NN, 2-Opt)** at n=10/12/15, on 50 instances each. Held-Karp
is feasible at these sizes; Active Search is per-instance iterative (5000 steps), which is why N
is small. NN/2-Opt/Held-Karp run on the *same* 50 instances so the mean +/- std is fair.

First: compute Held-Karp optimal tours for the comparison sets (`make_supervised`).

Tune in `python/compare_sizes.py`: `NUM_INSTANCES` (default 50), `NUM_STEPS` (default 5000),
`SIZES`.

In [ ]:
%%bash
cd /content/neural-tsp/cpp
set -e
g++ -std=c++17 -O2 make_supervised.cpp -o make_supervised

# Held-Karp optimal tours for the comparison sets (feasible: n<=15)
for n in 10 12 15; do
  ./make_supervised < ../data/comp_n${n}.txt > ../data/comp_opt_n${n}.txt
done

echo Done: comp_opt files written


In [ ]:
%cd /content/neural-tsp/python
%run compare_sizes.py

## 9 - Save the size-comparison figure + table to Drive

In [ ]:
import shutil, os

out_dir = '/content/drive/MyDrive/neural-tsp'
os.makedirs(out_dir, exist_ok=True)
for f in ['size_comparison.png', 'size_comparison.csv']:
    src = f'/content/neural-tsp/data/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{out_dir}/{f}')
        print('Saved', f, '->', out_dir)

## 10 - Final backup to Drive

In [ ]:
import shutil, os

for f in ['actor.pt', 'critic.pt']:
    src = f'/content/neural-tsp/python/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/drive/MyDrive/neural-tsp/python/{f}')

ckpt_dir = '/content/neural-tsp/python/checkpoints'
if os.path.exists(ckpt_dir):
    shutil.copytree(ckpt_dir, '/content/drive/MyDrive/neural-tsp/python/checkpoints', dirs_exist_ok=True)

shutil.copytree('/content/neural-tsp/data', '/content/drive/MyDrive/neural-tsp/data', dirs_exist_ok=True)
print('Everything saved to Google Drive.')

---

## Resume after disconnection

If Colab disconnects during RL training, run these to resume from the last checkpoint.

### Resume 1 - Re-mount & copy project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/drive/MyDrive/neural-tsp /content/neural-tsp
%cd /content/neural-tsp/python

### Resume 2 - Check available checkpoints

In [ ]:
import os

if os.path.exists('checkpoints'):
    ckpts = sorted([f for f in os.listdir('checkpoints') if f.endswith('.pt')],
                   key=lambda f: int(f.replace('ckpt_epoch','').replace('.pt','')))
    print(f'Available checkpoints: {ckpts}')
    print(f'Latest: {ckpts[-1]}')
else:
    print('No checkpoints found.')

### Resume 3 - Continue RL training

`train_rl.py` auto-detects the latest checkpoint (sorted by epoch number) and resumes from it.

In [ ]:
%run train_rl.py

---

## Scaling notes

- **Active Search cost** scales as `N_instances * NUM_STEPS`. 50 instances x 5000 steps at
  n=10/12/15 is ~1.5-2 h on a T4. Drop `NUM_STEPS` to ~2000 in `compare_sizes.py` for ~2.5x
  speed at a small quality cost, or raise `NUM_INSTANCES` for tighter error bars.
- **Larger n in the comparison**: Held-Karp is infeasible beyond ~n=18-20, so the size
  comparison is capped at n=15. To train/evaluate the *policy itself* at a larger n (e.g. n=20
  or n=50, RL-only), regenerate `train_rl.txt` / `eval.txt` at that n and re-run from step 4 -
  no code changes needed.